# Gated Recurrent Unit: Entrenamiento y Evaluación

In [72]:
import os
import random
import sys
import warnings
from pathlib import Path
import optuna
import time

import numpy as np
import pandas as pd
import matplotlib.dates as mdates
import torch
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.models.gru_wrapper import GRUModel
from src.eval.scorer import VolatilityScorer
from src.trainer.walk_forward import WalkForwardTrainer
from src.data.dataset import TimeSeriesDataset
from src.utils import trainer_toolkit as tt
from src.utils import ts_toolkit as tst
from src.utils import test_toolkit as ts

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

OUT_PATH = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROJECT_ROOT / "results" / "gru"
MODELS_PATH = PROJECT_ROOT / "models"

RESULTS_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)

df_train = pd.read_parquet(OUT_PATH / "brent_train.parquet")
y_train = df_train["log_returns"].values.astype("float64")
rs_train = df_train["rs_daily"].values.astype("float64")

df_test = pd.read_parquet(OUT_PATH / "brent_test.parquet")
y_test = df_test["log_returns"].values.astype("float64")
rs_test = df_test["rs_daily"].values.astype("float64")

assert not np.isnan(y_train).any() and not np.isnan(rs_train).any(), "NaNs en train"
assert not np.isnan(y_test).any() and not np.isnan(rs_test).any(), "NaNs en test"
assert df_train.index.max() < df_test.index.min(), "Solapamiento temporal train/test"

print("=" * 70)
print(f"Train: {len(y_train)} obs  ({df_train.index[0].date()} -> {df_train.index[-1].date()})")
print(f"Test:  {len(y_test)} obs  ({df_test.index[0].date()} -> {df_test.index[-1].date()})")
print("=" * 70)

Train: 2171 obs  (2014-01-03 -> 2022-11-03)
Test:  953 obs  (2022-11-07 -> 2026-08-21)


:::{admonition} Resumen
:class: info
- El protocolo Walk-Forward seleccionó una arquitectura GRU con ventana de 21 días, 64 neuronas y 1 capa.
- La asimetría de la pérdida QLIKE indujo un sesgo defensivo en la red, provocando una sobreestimación sistemática de la volatilidad base durante los periodos de calma.
- La naturaleza recurrente de las compuertas generó un efecto de inercia temporal, suavizando la respuesta del modelo y retrasando la detección de los picos de volatilidad.
- Fuera de la muestra de entrenamiento, el modelo mostró menor capacidad predictiva que los modelos paramétricos y dejó efectos ARCH remanentes a corto plazo.
- A pesar de su flexibilidad no paramétrica, la GRU subestimó la magnitud del shock extremo de 2026, evidenciando limitaciones para reaccionar de forma inmediata ante cambios abruptos de régimen.
:::

## Introducción

La Unidad Recurrente con Compuertas (*Gated Recurrent Unit*, GRU) es una arquitectura de red neuronal recurrente diseñada para modelar dependencias temporales en datos secuenciales. A diferencia de las redes neuronales tradicionales, incorpora mecanismos de memoria que permiten conservar y actualizar información de observaciones anteriores mediante compuertas entrenables.

En el contexto de las series financieras, esta arquitectura ofrece una alternativa a los modelos econométricos tradicionales, al permitir que la relación entre los retornos históricos y la volatilidad futura sea aprendida directamente a partir de los datos. Su estructura recurrente facilita la identificación de patrones temporales y cambios en la dinámica de los retornos, sin establecer previamente una ecuación paramétrica para la evolución de la varianza condicional.

El objetivo de esta sección es presentar, entrenar y evaluar un modelo GRU para la predicción de la volatilidad del crudo Brent bajo el protocolo del benchmark. Se estudia el efecto de distintas longitudes de secuencia sobre el desempeño predictivo de la red, manteniendo las condiciones de información y evaluación establecidas para los demás modelos del proyecto.

### Arquitectura

Una red GRU procesa secuencialmente las observaciones de entrada y actualiza un estado oculto que resume la información temporal disponible. Para controlar este proceso, utiliza dos mecanismos principales: la compuerta de actualización (*update gate*) y la compuerta de reinicio (*reset gate*).

La compuerta de actualización determina qué proporción de la información anterior debe conservarse, mientras que la compuerta de reinicio regula la influencia del estado previo en la construcción de nueva información. Para una observación de entrada $\mathbf{x}_t$, estas operaciones se expresan como:

$$
\mathbf{z}_t = \sigma\left(\mathbf{W}_z\mathbf{x}_t+\mathbf{U}_z\mathbf{h}_{t-1}+\mathbf{b}_z\right),
$$

$$
\mathbf{r}_t = \sigma\left(\mathbf{W}_r\mathbf{x}_t+\mathbf{U}_r\mathbf{h}_{t-1}+\mathbf{b}_r\right),
$$

donde $\mathbf{z}_t$ representa la compuerta de actualización, $\mathbf{r}_t$ la compuerta de reinicio, $\sigma$ la función sigmoide y $\mathbf{h}_{t-1}$ el estado oculto anterior.

El estado candidato se calcula mediante:

$$
\tilde{\mathbf{h}}_t =
\tanh\left(
\mathbf{W}_h\mathbf{x}_t+
\mathbf{U}_h(\mathbf{r}_t\odot\mathbf{h}_{t-1})+
\mathbf{b}_h
\right),
$$

y el nuevo estado oculto se obtiene como:

$$
\mathbf{h}_t =
(1-\mathbf{z}_t)\odot\mathbf{h}_{t-1}
+
\mathbf{z}_t\odot\tilde{\mathbf{h}}_t,
$$

donde $\odot$ representa el producto de Hadamard (elemento a elemento). 

Una vez calculado el estado oculto final de la secuencia, $\mathbf{h}_t \in \mathbb{R}^{H}$ (donde $H$ es el número de unidades ocultas), este vector resume la información de la ventana temporal. Para generar un pronóstico escalar de la varianza condicional, este vector se proyecta a través de una capa densa (*fully connected layer*) con pesos $\mathbf{w} \in \mathbb{R}^{H}$ y sesgo $b \in \mathbb{R}$:

$$
y_t = \mathbf{w}^{\top}\mathbf{h}_t + b.
$$

Para garantizar que la varianza pronosticada sea estrictamente positiva (un requisito matemático para la volatilidad), el resultado escalar $y_t$ pasa por una función de activación *Softplus*, la cual es una versión suavizada de la función ReLU:

$$
\hat{\sigma}_t^2 = \text{Softplus}(y_t) = \ln(1 + e^{y_t}),
$$

donde $\boldsymbol{\theta}$ agrupa los pesos y sesgos tanto de las unidades GRU como de la capa de salida.

Estos mecanismos permiten que la red aprenda qué información temporal conservar, modificar o descartar. A diferencia del perceptrón multicapa (*Multilayer Perceptron*, MLP), que recibe los rezagos como un vector de características, la GRU procesa las observaciones como una secuencia ordenada, incorporando explícitamente la dependencia temporal en su arquitectura.

Para aplicar esta arquitectura al pronóstico de la volatilidad del crudo Brent, sea $r_t$ el retorno logarítmico diario. Para una ventana temporal de longitud $L$, la secuencia de entrada se define como:

$$
\mathbf{X}_t =
\begin{bmatrix}
r_{t-L}\\
r_{t-L+1}\\
\vdots\\
r_{t-1}
\end{bmatrix}
\in \mathbb{R}^{L\times 1}.
$$

El modelo completo (la GRU y su capa de salida) genera el pronóstico de la varianza condicional del día inmediatamente posterior mediante la función $f_{\boldsymbol{\theta}}(\mathbf{X}_t)$.

El problema se aborda mediante una estrategia de predicción de un paso hacia adelante (*one-step-ahead forecasting*). El entrenamiento consiste en encontrar los parámetros $\boldsymbol{\theta}$ que minimicen la pérdida QLIKE sobre las observaciones disponibles, dado que esta métrica penaliza asimétricamente las subestimaciones de la volatilidad. El problema de optimización se expresa como:

$$
\boldsymbol{\theta}^{*}
=\arg\min_{\boldsymbol{\theta}}
\frac{1}{N}
\sum_{i=1}^{N}
\mathcal{L}_{\text{QLIKE}}
\left(
\sigma_{i}^{2},
\hat{\sigma}_{i}^{2}
\right),
$$

donde $N$ es el número de secuencias de entrenamiento.

Los parámetros se actualizan mediante retropropagación a través del tiempo (*backpropagation through time*, BPTT) y el optimizador Adaptive Moment Estimation (Adam). La elección del número de unidades recurrentes, las capas ocultas, la longitud de las secuencias ($L$) y los parámetros de entrenamiento (tasa de aprendizaje, *dropout*) influye directamente sobre la capacidad de ajuste y generalización del modelo.

La implementación de la GRU se realiza utilizando la librería `PyTorch` en su versión `2.13.0`, aprovechando su módulo de diferenciación automática para el cálculo de gradientes y la optimización de los pesos.

## Entrenamiento

Para la selección de hiperparámetros del modelo GRU, se mantuvo el mismo protocolo experimental (Walk-Forward) y el mismo presupuesto computacional que en los modelos paramétricos. A diferencia del GARCH y el EGARCH, la búsqueda de Optuna exploró la arquitectura de la red neuronal y sus parámetros de regularización.

Para adaptar la serie temporal al formato requerido por la red, se construyó una matriz de rezagos (*lag matrix*) para cada ventana del esquema Walk-Forward. Siguiendo los principios de validación cruzada para series temporales (`timeseriescv`), cada fila de esta matriz contenía una secuencia histórica de longitud `look_back`, la cual se utilizó como vector de entrada para predecir el paso inmediatamente posterior. Este enfoque garantizó la integridad temporal de los datos y evitó cualquier tipo de fuga de información futura (*look-ahead bias*) durante el entrenamiento.

El espacio de búsqueda incluyó la longitud de la ventana de entrada (`look_back`), el número de neuronas en la capa oculta (`hidden_size`), la cantidad de capas recurrentes, la tasa de abandono (`dropout`) y la tasa de aprendizaje. Para mitigar el sobreajuste, se incorporó también la penalización $\ell_2$ (`weight_decay`). El entrenamiento se configuró con un máximo de 200 épocas y una paciencia de 20 para la parada temprana interna de la red. El desempeño se siguió evaluando mediante la pérdida QLIKE sobre los conjuntos de validación.

In [ ]:
def gru_search_space(trial: optuna.Trial) -> dict:
    return {
        "look_back": trial.suggest_categorical("look_back", [7, 14, 21, 28]),
        "hidden_size": trial.suggest_categorical("hidden_size", [32, 64, 128]),
        "num_layers": trial.suggest_int("num_layers", 1, 2),
        "dropout": trial.suggest_float("dropout", 0.0, 0.3),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        "max_epochs": 200,
        "patience": 20,
        "retrain_freq": 1,
        "seed": 42
    }

exec_id = 1
current_warmup = 1200 + (exec_id * 10)

exec_dir = RESULTS_PATH / f"exec_{exec_id}"
exec_dir.mkdir(parents=True, exist_ok=True)

trainer = WalkForwardTrainer(
    model_class=GRUModel,
    scorer=VolatilityScorer(),
    output_dir=exec_dir,
    n_splits=5, warmup=current_warmup,
    embargo=1, n_trials=50,
    suggest_params_fn=gru_search_space,
    early_stop_rounds=15
)

trainer.run(y_train, rs_train)

Respecto a la eficiencia del algoritmo de búsqueda, esta alcanzó el límite máximo de 50 evaluaciones, agotando el presupuesto computacional asignado. A diferencia de los modelos paramétricos, el mecanismo de poda (*Pruning*) no descartó ninguna configuración (0 podados), lo que indicó que todas las arquitecturas evaluadas lograron completar las particiones de validación sin inestabilidad numérica. La mejor configuración registró un QLIKE de 0.6413.

El entrenamiento bajo el esquema Walk-Forward tuvo una duración de 7.2 minutos (430.66 segundos), con un tiempo medio de 1.72 segundos por pliegue. Este incremento respecto a los 9.12 y 12.54 segundos de los modelos GARCH y EGARCH reflejó el costo inherente a los métodos iterativos (descenso de gradiente) de las redes neuronales frente a las estimaciones cerradas de máxima verosimilitud. Este mayor tiempo de cómputo fue el compromiso necesario para dotar al modelo de mayor flexibilidad no paramétrica.

In [7]:
resumen_data = []

exec_dir = RESULTS_PATH

tiempo_total_rep = 0.0
tiempo_promedio_fold = 0.0
num_folds = 0
total_trials, completos, podados, fallidos = 0, 0, 0, 0
mejor_qlike = "-"

csv_files = list(exec_dir.glob("*_results.csv"))
if csv_files:
    df_csv = pd.read_csv(csv_files[0])
    tiempo_total_rep = df_csv["train_time_sec"].sum()
    tiempo_promedio_fold = df_csv["train_time_sec"].mean()
    num_folds = len(df_csv)
    
db_path = exec_dir / "optuna_study.db"
if db_path.exists():
    try:
        storage_url = f"sqlite:///{db_path.resolve()}"
        study = optuna.load_study(study_name="GRUModel_study", storage=storage_url)

        total_trials = len(study.trials)
        completos = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
        podados  = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
        fallidos = len([t for t in study.trials if t.state == optuna.trial.TrialState.FAIL])

        mejor_qlike = round(study.best_value, 4)

    except Exception as e:
        print(f"Error leyendo BD en {exec_dir}: {e}")

resumen_data.append({
    "Total Trials": total_trials,
    "Completos": completos,
    "Podados": podados,
    "Mejor QLIKE": mejor_qlike,
    "Tiempo Total (s)": round(tiempo_total_rep, 2),
    "Tiempo por Fold (s)": round(tiempo_promedio_fold, 3)
})

df_search_params = pd.DataFrame(resumen_data)
tt.display_styled_df(df=df_search_params, decimals=4, align_left_first=False)

Total Trials,Completos,Podados,Mejor QLIKE,Tiempo Total (s),Tiempo por Fold (s)
50,50,0,0.6413,430.6600,1.7230


El ranking de configuraciones muestra que las cinco mejores opciones presentaron una pérdida QLIKE en el rango de 0.641 a 0.717. Las diferencias entre estas configuraciones fueron menores que la alta variabilidad entre folds ($\sigma > 0.25$), lo que indicó un empate estadístico en su desempeño predictivo. A pesar de que los hiperparámetros continuos (tasa de aprendizaje, *dropout* y regularización L2) variaron entre las cinco opciones, se observó un consenso en la arquitectura base: todas seleccionaron una ventana de memoria (*lookback*) de 21 días, 64 neuronas ocultas y una sola capa recurrente.

Esta convergencia arquitectónica aporta información sobre la dinámica de los datos. Aunque el espacio de búsqueda exploró ventanas temporales más cortas (7 y 14 días) y más largas (28 días), la red neuronal convergió de forma consistente hacia los 21 días. Este valor equivale aproximadamente a un mes de días de negociación, sugiriendo que este es el horizonte de memoria óptimo para capturar la persistencia de la volatilidad del Brent sin incorporar ruido de periodos excesivamente largos. De igual forma, la elección de una sola capa con 64 neuronas evidenció que la red logró un equilibrio entre capacidad de representación y parsimonia, rechazando estructuras más profundas o de mayor tamaño que habrían incrementado el riesgo de sobreajuste.

In [11]:
df_gru = pd.read_csv(RESULTS_PATH / "GRUModel_results.csv")

df_gru["lr"] = df_gru["learning_rate"].apply(lambda x: f"{x:.4f}")
df_gru["wd"] = df_gru["weight_decay"].apply(lambda x: f"{x:.1e}")

cols_agrupar = ["look_back", "hidden_size", "num_layers", "dropout", "lr", "wd"]
df_agg = df_gru.groupby(cols_agrupar)["qlike"].agg(["mean", "std"]).reset_index()
df_agg["QLIKE"] = df_agg.apply(lambda x: f"{x['mean']:.3f} ± {x['std']:.3f}", axis=1)
df_top5 = df_agg.sort_values(by="mean").head(5)

df_top5 = df_top5.rename(columns={
    "look_back": "Lookback",
    "hidden_size": "Neuronas",
    "num_layers": "Capas",
    "dropout": "Dropout",
    "lr": "LR",
    "wd": "Weight Decay"
})

df_final = df_top5[["Lookback", "Neuronas", "Capas", "Dropout", "LR", "Weight Decay", "QLIKE"]]

tt.display_styled_df(df=df_final, align_left_first=False, decimals=3)

Lookback,Neuronas,Capas,Dropout,LR,Weight Decay,QLIKE
21,64,1,0.223,0.0058,1.2e-06,0.641 ± 0.298
21,64,1,0.294,0.0043,6.1e-06,0.686 ± 0.274
21,64,1,0.299,0.0053,1.0e-06,0.715 ± 0.335
21,64,1,0.278,0.0018,1.2e-06,0.717 ± 0.435
21,64,1,0.274,0.0044,1.2e-06,0.717 ± 0.252


Las dos configuraciones con menor pérdida compartieron la misma arquitectura base (Lookback de 21 días, 64 neuronas y 1 capa), difiriendo únicamente en sus hiperparámetros de regularización y tasa de aprendizaje. La diferencia entre sus medias ($\Delta \text{QLIKE} = 0.045$) fue menor que la variabilidad observada entre los folds ($\sigma > 0.27$), lo que indicó que ambas fueron indistinguibles en desempeño predictivo.

Un hallazgo relevante fue la magnitud de la desviación estándar del QLIKE en el GRU ($\sigma \approx 0.30$), la cual superó holgadamente la reportada por los modelos GARCH y EGARCH ($\sigma \approx 0.06$ y $0.10$, respectivamente). Una hipótesis plausible para explicar este fenómeno radica en la naturaleza de los algoritmos de optimización. Mientras que el GARCH utiliza una estimación determinista de Máxima Verosimilitud que converge a un único óptimo, el GRU se entrena mediante descenso de gradiente estocástico sobre una superficie de pérdida no convexa. Esto, sumado a la inicialización aleatoria de los pesos en cada partición (folds), induce una mayor varianza en los resultados. Adicionalmente, la alta flexibilidad de la red neuronal provocó que su desempeño fuera más volátil ante los distintos regímenes de volatilidad presentes en cada fold de validación.

Ante la equivalencia predictiva en la media, se aplicó el criterio de parsimonia y estabilidad. Se seleccionó la primera configuración, la cual alcanzó el menor QLIKE medio y utilizó un *dropout* de 0.223 con una tasa de aprendizaje de 0.0058, representando la variante más estable dentro de la arquitectura dominante.

In [20]:
best_model = GRUModel(
    look_back=21, hidden_size=64,
    num_layers=1, dropout=0.223, learning_rate=0.0058,
    weight_decay=1.2e-06, max_epochs=200, patience=20, retrain_freq=1, seed=42
)

best_model.fit(y_train, rs_train)

El análisis de la curva de aprendizaje descarta la presencia de sobreajuste. En un escenario de sobreajuste clásico, la pérdida de entrenamiento continuaría disminuyendo mientras la de validación empezaría a incrementarse, abriendo una brecha progresiva. En este caso, la pérdida de validación no solo se mantiene estable, sino que es consistentemente menor que la de entrenamiento desde el inicio. Esta brecha negativa se explica por el sesgo de selección temporal del conjunto de datos: la partición cronológica destina el periodo de mayor dificultad predictiva (los shocks extremos de 2020) al entrenamiento, mientras que la validación (2021-2022) corresponde a un régimen de volatilidad más moderado y, por tanto, con un QLIKE intrínsecamente más bajo.

In [21]:
tt.plot_learning_curve(
    history=best_model.get_history(), 
    title="Curva de Aprendizaje",
    ylab = "Quasi-Likelihood (QLIKE)",
    xlab_pad=14, ylab_pad=14,
    line_width=2.0,
    title_size=14, label_size=14, 
    tick_size=12, legend_size=14,
    ylim=(0, 5), xlim=(0, 24), y_step=0.5, x_step=1,
    legend_bbox=(0.95, 0.95)
)

El modelo alcanzó su punto de mínimo error en la época 3. Esta convergencia prematura sugiere que la red encontró rápidamente un mínimo local que no pudo superar en las 20 épocas posteriores, a pesar de la paciencia configurada. Este comportamiento, sumado a la baja pérdida inicial en validación, indica que la arquitectura seleccionada (una sola capa y 64 neuronas) opera cerca de su techo de capacidad para este conjunto de datos. La red aprende la tendencia central de la varianza condicional rápidamente, pero no logra refinar la predicción de los picos de volatilidad, lo cual es consistente con el desempeño general del modelo no paramétrico.

Para evaluar el ajuste dentro de la muestra de entrenamiento, a diferencia de los modelos paramétricos, no es posible extraer los residuos directamente del objeto del modelo. En su lugar, se realizó un pase hacia adelante (*forward pass*) utilizando la totalidad de la serie de entrenamiento. Mediante la clase `TimeSeriesDataset`, se construyó la matriz de rezagos correspondiente, agrupando los retornos en ventanas históricas de 21 días. Estas ventanas se procesaron en un solo lote a través de la red neuronal en modo de evaluación (`eval()`), desactivando el cálculo de gradientes y las capas de regularización. Las varianzas pronosticadas se alinearon temporalmente con los datos reales, descartando los primeros 21 días de la serie (`y_train[look_back:]`) para garantizar que cada predicción correspondiera exactamente con su valor observado.

In [44]:
best_model.net.eval()
look_back = best_model.context_length

train_ds = TimeSeriesDataset(y_train, rs_train * best_model.target_scale, look_back)
train_loader = DataLoader(train_ds, batch_size=len(train_ds), shuffle=False)

with torch.no_grad():
    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(best_model.device)
        preds_train_var = best_model.net(x_batch).cpu().numpy().flatten()

preds_train_var = preds_train_var / best_model.target_scale
y_train_aligned = y_train[look_back:]
rs_train_aligned = rs_train[look_back:]

Para evaluar la estructura de los residuos tanto en el conjunto de entrenamiento (In-Sample) como en el de prueba (Out-of-Sample), a partir de las varianzas predichas y los retornos reales, se calcularon los residuos estandarizados mediante la siguiente expresión:

$$
z_t = \frac{y_t}{\sqrt{\hat{\sigma}_t^2}}
$$

El análisis de los residuos sobre el conjunto de entrenamiento reveló un ajuste estructural casi perfecto. Las pruebas de Ljung-Box (autocorrelación), ARCH-LM (heterocedasticidad) y BDS (dependencia no lineal) no rechazaron sus hipótesis nulas ($p > .05$). Lo anterior indicó que la red logró absorber la totalidad de la dependencia temporal y los efectos condicionales dentro de la muestra. Por otro lado, las pruebas de Jarque-Bera y Lilliefors rechazaron la hipótesis nula de normalidad ($p < .001$), evidenciando la presencia de colas pesadas no capturadas por la estructura gaussina. 

In [42]:
std_residuals_train = y_train_aligned / np.sqrt(preds_train_var)
std_residuals_train = std_residuals_train[np.isfinite(std_residuals_train)]

df_diag = tt.run_model_diagnostics(
    std_residuals=std_residuals_train,
    lags=(10, 20, 30),
    p_adjust="bonferroni",
    alpha=0.05,
    english=False,
    column_titles=(
        "Prueba Estadística", "Estadístico",
        "p-valor", "p-adj", "Conclusión (α = 0.05)")
)

df_display = df_diag.copy()

df_display["Estadístico"] = df_display["Estadístico"].apply(
    lambda x: f"{x:.4f}" if pd.notna(x) else "--")

def _format_p(x):
    if pd.isna(x): return "--"
    if x < 0.001: return "< .001"
    formatted = f"{x:.3f}"
    return formatted[1:] if formatted.startswith("0.") else formatted

df_display["p-valor"] = df_display["p-valor"].apply(_format_p)
df_display["p-adj"] = df_display["p-adj"].apply(_format_p)

tt.display_styled_df(df=df_display, p_value_cols=[])

Prueba Estadística,Estadístico,p-valor,p-adj,Conclusión (α = 0.05)
Ljung-Box (Lag 10),11.4486,.324,.971,Sin Autocorrelación
Ljung-Box (Lag 20),30.2633,.066,.197,Sin Autocorrelación
Ljung-Box (Lag 30),39.5048,.115,.344,Sin Autocorrelación
ARCH-LM (Lag 10),2.4686,.991,1.000,Sin Efectos ARCH
ARCH-LM (Lag 20),2.7096,1.000,1.000,Sin Efectos ARCH
ARCH-LM (Lag 30),3.2944,1.000,1.000,Sin Efectos ARCH
Jarque-Bera,45161.2037,< .001,--,Residuos No Normales
Lilliefors,0.0574,< .001,--,Residuos No Normales
BDS (Dim 2),0.8345,.404,--,Sin Estructura No Lineal


Tal como ocurrió con los modelos econométricos, los residuos estandarizados del GRU presentan una estructura consistente con una distribución de colas pesadas, ajustándose a una distribución $t$ de Student. Esta correspondencia se observa en el gráfico Q-Q, donde los cuantiles empíricos siguen la línea de referencia diagonal a lo largo de casi toda la distribución. No obstante, se identifican dos puntos atípicos en el extremo superior de la cola derecha que se desvían de la recta teórica, lo que indica que la distribución $t$ ($\nu = 4.5672$) no logra modelar la magnitud de los eventos positivos más extremos. A pesar de estas dos desviaciones puntuales, las pruebas formales de bondad de ajuste (Kolmogorov-Smirnov: $p = .9162$; Cramér-von Mises: $p = .8985$) no rechazan la hipótesis de pertenencia a la distribución $t$.

In [48]:
_, _ = tst.plot_distribution_fit(
    pd.Series(std_residuals_train),
    dist="t",
    titles=("Distribución de Residuos", "Gráfico Q-Q (t-Student)"),
    xlab="Residuo estandarizado",
    ylab_left="Densidad",
    xlab_right="Cuantiles teóricos",
    ylab_right="Cuantiles muestrales",
    ylim_left=(0, 0.5),
    legend_loc="upper left"
)

La LRT rechazó la hipótesis nula de igualdad de ajuste ($p < .001$). Los criterios de información evidenciaron una reducción de $\Delta\text{AIC} = 383.11$ y $\Delta\text{BIC} = 377.44$ a favor de la distribución $t$. Esto indicó que la inclusión del parámetro de grados de libertad ($\nu \approx 4.57$) mejoró la descripción de las colas pesadas de los residuos en la muestra de entrenamiento en comparación con el modelo Gaussiano. 

Es importante precisar que, a diferencia de los modelos GARCH y EGARCH, el GRU no asume una distribución paramétrica específica durante su estimación, dado que su función de pérdida (QLIKE) se optimiza directamente sobre la varianza. Por tanto, este contraste estadístico no valida un supuesto interno del modelo, sino que confirma que la estructura empírica de los residuos del GRU presenta colas pesadas que la distribución Normal no captura de forma adecuada.

In [49]:
_= tst.distribution_fit_table(
    pd.Series(std_residuals_train),
    column_titles=(
        "Distribución", 
        "Log-Likelihood", 
        "AIC", 
        "BIC", 
        "Estadístico LRT", 
        "p-valor LRT"
    )
)

Distribución,Log-Likelihood,AIC,BIC,Estadístico LRT,p-valor LRT
Normal,"-3,893.28","7,790.55","7,801.90",-,-
t-Student,"-3,700.72","7,407.44","7,424.46",385.11,< .001


A diferencia de los modelos paramétricos GARCH y EGARCH, los cuales presentaron dificultades para capturar la magnitud de los shocks extremos, el modelo GRU capturó los picos de volatilidad de forma casi perfecta. La red neuronal logró seguir la trayectoria de la volatilidad realizada, reproduciendo el nivel de eventos extremos como la crisis del COVID-19 en 2020 y los picos de 2022. Esta correspondencia visual evidencia la flexibilidad de la arquitectura no paramétrica para ajustarse a dinámicas condicionales complejas.

In [41]:
look_back = best_model.context_length

dates_aligned = df_train.index[look_back:]
rs_volatilidad_aligned = np.sqrt(rs_train[look_back:])
pred_volatilidad = np.sqrt(preds_train_var) 

mask_crisis = (dates_aligned >= "2020-01-01") & (dates_aligned <= "2020-06-30")

ts.plot_volatility_fit(
    dates=dates_aligned,
    rs_variance=rs_volatilidad_aligned,      
    pred_variance=pred_volatilidad,
    title="Ajuste dentro la Muestra de Entrenamiento (GRU)", 
    ylab="Volatilidad",              
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$", 
    label_pred=r"Volatilidad GRU $\left(\hat{\sigma}_t\right)$",
    ylim=(-0.01, 0.35), 
    legend_bbox=(0.01, 0.98), 
    zoom_slice=mask_crisis, 
    zoom_pad_frac=0.04,
    zoom_title="Zoom: Crisis COVID-19",
    zoom_bbox=(0.18, 0.35, 0.40, 0.40),
    title_size=14, label_size=14, tick_size=12, legend_size=14
)

Este alto nivel de ajuste es un resultado esperado dentro de la muestra de entrenamiento. Al ser evaluada con los mismos datos con los que se optimizaron sus pesos, la red neuronal despliega toda su capacidad para mapear y memorizar la secuencia histórica. Por lo tanto, para determinar si la arquitectura posee capacidad de generalización ante nuevos regímenes de mercado, se procedió a la evaluación sobre el conjunto de prueba (Out-of-Sample).

## Evaluación

La evaluación del rendimiento de un modelo de volatilidad requiere contrastar su capacidad de ajuste sobre los datos observados con su capacidad predictiva sobre datos no vistos. Mientras que el conjunto de entrenamiento (In-Sample) permite verificar la convergencia del algoritmo a la dinámica histórica, la evaluación Out-of-Sample (conjunto de prueba) determina la utilidad real del modelo ante nuevos regímenes de mercado. 

Para diagnosticar la presencia de sobreajuste (*overfitting*) o subajuste (*underfitting*), se aplicaron los criterios pre-registrados en el protocolo experimental. Estos criterios se basaron en la diferencia absoluta entre las métricas de prueba y entrenamiento, denotada como:

$$
\Delta_{abs} = M_{test} - M_{train}
$$

Se estableció un umbral de tolerancia absoluto de $|\Delta_{abs}| = 0.05$ para todas las métricas de evaluación. Un valor de $\Delta_{abs}$ que superara este margen evidenciaría falta de generalización. 

No obstante, en el contexto de los modelos de volatilidad, es fundamental resaltar la naturaleza asimétrica de la pérdida QLIKE. Dado que esta métrica penaliza severamente la subestimación de la volatilidad, pero es indulgente con la sobreestimación, la interpretación tradicional del sobreajuste debe matizarse. Un modelo que desarrolla un sesgo defensivo (sobreestimando la varianza de forma constante) puede presentar un QLIKE de entrenamiento superior al de prueba. Esto no indica una mejora en la generalización, sino que refleja que el conjunto de prueba presentó regímenes de volatilidad que se acoplaron mejor a dicha sobreestimación. Por lo tanto, el análisis de la brecha $\Delta_{abs}$ se complementó con la inspección de las trayectorias predichas y los diagnósticos de residuos.

Las predicciones fuera de muestra se obtuvieron ejecutando el método `forecast` del wrapper del modelo. Este procedimiento aplicó un esquema de ventana expansiva (*rolling forward*) con reentrenamiento periódico, garantizando la ausencia de fuga de información. Las varianzas condicionales predichas se generaron en la escala original para permitir su comparación directa con la varianza realizada.

In [ ]:
start_time = time.time()
preds_test = best_model.forecast(y_seq=y_test, rs_seq=rs_test) 
preds_time = time.time() - start_time

df_preds = pd.DataFrame({
    "date": df_test.index, 
    "rs_true": rs_test, 
    "y_pred": preds_test
})

output_path = RESULTS_PATH / "gru_test_predictions.csv"
df_preds.to_csv(output_path, index=False)

La generación de pronósticos fuera de muestra mediante el esquema de ventana expansiva y reentrenamiento periódico tuvo una duración total de 13.71 minutos. Esto equivale a un tiempo medio de aproximadamente 0.86 segundos por predicción (para las 953 observaciones del conjunto de prueba). Este costo es sustancialmente superior al de los modelos paramétricos, cuyo tiempo de inferencia es de milisegundos, reflejando la carga computacional inherente a la actualización iterativa de los pesos en las arquitecturas de Deep Learning.

El análisis de los residuos fuera de muestra presentó un patrón consistente con el observado en el modelo EGARCH. Las pruebas de Ljung-Box (autocorrelación) y BDS (dependencia no lineal) no rechazaron sus hipótesis nulas ($p > .05$), indicando que el GRU capturó la estructura temporal de largo plazo. Sin embargo, la prueba ARCH-LM detectó efectos ARCH significativos en el rezago 10 ($p\text{-adj} = .005$), aunque no en los rezagos 20 ($p\text{-adj} = .056$) ni 30 ($p\text{-adj} = .403$). 

In [51]:
preds_path = RESULTS_PATH / "gru_test_predictions.csv"
df_preds_test = pd.read_csv(preds_path)
preds_test_var = df_preds_test["y_pred"].values
std_residuals_oos = y_test / np.sqrt(preds_test_var)

std_residuals_oos = std_residuals_oos[np.isfinite(std_residuals_oos)]

df_diag = tt.run_model_diagnostics(
    std_residuals=std_residuals_oos,
    lags=(10, 20, 30),
    p_adjust="bonferroni",
    alpha=0.05,
    english=False,
    column_titles=(
        "Prueba Estadística", "Estadístico",
        "p-valor", "p-adj", "Conclusión (α = 0.05)")
)

df_display = df_diag.copy()

df_display["Estadístico"] = df_display["Estadístico"].apply(
    lambda x: f"{x:.4f}" if pd.notna(x) else "--")

def _format_p(x):
    if pd.isna(x): return "--"
    if x < 0.001: return "< .001"
    formatted = f"{x:.3f}"
    return formatted[1:] if formatted.startswith("0.") else formatted

df_display["p-valor"] = df_display["p-valor"].apply(_format_p)
df_display["p-adj"] = df_display["p-adj"].apply(_format_p)

tt.display_styled_df(df=df_display, p_value_cols=[])

Prueba Estadística,Estadístico,p-valor,p-adj,Conclusión (α = 0.05)
Ljung-Box (Lag 10),10.1027,.432,1.000,Sin Autocorrelación
Ljung-Box (Lag 20),25.2667,.191,.574,Sin Autocorrelación
Ljung-Box (Lag 30),28.7093,.533,1.000,Sin Autocorrelación
ARCH-LM (Lag 10),28.0636,.002,.005,Efectos ARCH Significativos
ARCH-LM (Lag 20),35.2500,.019,.056,Sin Efectos ARCH
ARCH-LM (Lag 30),38.6204,.134,.403,Sin Efectos ARCH
Jarque-Bera,1691.3608,< .001,--,Residuos No Normales
Lilliefors,0.0556,< .001,--,Residuos No Normales
BDS (Dim 2),0.5877,.557,--,Sin Estructura No Lineal


Este resultado evidencia que, al igual que el EGARCH, el GRU presentó dificultades para capturar los clústeres de volatilidad a corto plazo. La presencia de efectos ARCH en el rezago 10 indica que los residuos conservaron dependencia condicional durante los días inmediatamente posteriores a un shock. En el caso de la red neuronal, esto se relaciona con la inercia de la arquitectura recurrente: el modelo suaviza la respuesta y reacciona con un rezago temporal. Por lo tanto, no logra predecir de forma inmediata la agrupación de la volatilidad diaria tras un evento extremo, requiriendo varios pasos para asimilar el cambio de régimen.

El comportamiento de los residuos fuera de muestra fue similar al observado en el conjunto de entrenamiento. El ajuste a la distribución $t$ de Student arrojó un parámetro de grados de libertad de $\nu = 4.3015$. Al ser ligeramente inferior al obtenido In-Sample ($\nu \approx 4.57$), este resultado indicó que los residuos de prueba presentaron colas ligeramente más pesadas. Las pruebas formales de bondad de ajuste (Kolmogorov-Smirnov: $p = .9096$; Cramér-von Mises: $p = .8536$) no rechazaron la hipótesis de pertenencia a la distribución $t$. En el gráfico Q-Q, los cuantiles empíricos presentaron una alineación con la línea diagonal, con desviaciones puntuales en los valores extremos de las colas.

In [52]:
_, _ = tst.plot_distribution_fit(
    pd.Series(std_residuals_oos),
    dist="t",
    titles=("Distribución de Residuos", "Gráfico Q-Q (t-Student)"),
    xlab="Residuo estandarizado",
    ylab_left="Densidad",
    xlab_right="Cuantiles teóricos",
    ylab_right="Cuantiles muestrales",
    ylim_left=(0, 0.7),
    legend_loc="upper left"
)

La asimetría de la pérdida QLIKE condicionó el diagnóstico de sobreajuste en el GRU. El QLIKE en prueba (0.8383) fue inferior al de entrenamiento (3.0254). Tradicionalmente, una brecha de esta magnitud (con el error de prueba menor) descartaría el sobreajuste; sin embargo, este resultado se explicó por la asimetría de la métrica. Durante el entrenamiento, el modelo desarrolló un sesgo defensivo, sobreestimando la volatilidad base para evitar penalizaciones extremas. Al evaluarse en prueba, los regímenes de volatilidad se acercaron a esa línea base, reduciendo el error promedio. No obstante, al comparar el QLIKE de prueba (0.8383) con el de validación cruzada en Optuna (0.641 ± 0.298), se evidenció una pérdida de generalización.

In [74]:
look_back = best_model.context_length
rs_train_aligned = rs_train[look_back:] 

ts.display_train_test_comparison(
    rs_train=rs_train_aligned,
    preds_train=preds_train_var,
    rs_test=rs_test,
    preds_test=preds_test_var,
    column_titles=(
        "Métrica", 
        "Entrenamiento", 
        "Prueba", 
        "Diferencia"
    )
)

Métrica,Entrenamiento,Prueba,Diferencia
QLIKE,3.0254,0.8383,-2.1871
CRPS,0.0316,0.0127,-0.0188
Pinball (τ=0.05),0.0116,0.0027,-0.0090
Pinball (τ=0.95),0.0096,0.0021,-0.0075
MSE,0.0005,0.0000,-0.0005
RMSE,0.0221,0.0028,-0.0193
MAE,0.0178,0.0006,-0.0172
HMSE,0.9570,29.3712,+28.4142
HMAE,0.9781,1.2342,+0.2561


Las métricas probabilísticas (CRPS y Pinball) y los errores puntuales (MSE, RMSE, MAE) disminuyeron en prueba y se mantuvieron en rangos aceptables, indicando que la red capturó la tendencia central. Sin embargo, las métricas ajustadas por heterocedasticidad aumentaron: HMSE (+28.4142) y HMAE (+0.2561). El HMSE ponderó los errores por el inverso de la volatilidad, amplificando el error durante la turbulencia de 2026.

Este incremento del HMSE evidenció un subajuste en la dinámica de corto plazo. El modelo siguió el clúster general del mercado, pero presentó un efecto de inercia temporal: subestimó la magnitud del shock inmediato en 2026. La naturaleza recurrente de la arquitectura limitó la reacción abrupta, generando errores en los días de mayor volatilidad. El HMSE reflejó esta limitación: la GRU capturó la dinámica habitual, pero careció del sesgo inductivo necesario para adaptarse de inmediato a los cambios de régimen.

In [73]:
df_preds_test["date"] = pd.to_datetime(df_preds_test["date"])
mask_crisis = (df_preds_test["date"] >= "2026-02-01") & (df_preds_test["date"] <= "2026-06-30")

ts.plot_volatility_fit(
    dates=df_preds_test["date"],
    rs_variance=rs_test_vol,      
    pred_variance=pred_test_vol,
    title="Ajuste fuera de la Muestra de Entrenamiento (GRU)", 
    ylab="Volatilidad",              
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$", 
    label_pred=r"Volatilidad GRU $\left(\hat{\sigma}_t\right)$",
    ylim=(-0.01, 0.35), legend_bbox=(0.01, 0.98), 
    zoom_slice=mask_crisis, zoom_pad_frac=0.04,
    zoom_title="Zoom: Pico de Volatilidad (2026)",
    zoom_bbox=(0.25, 0.35, 0.40, 0.40),
    title_size=14, label_size=14, tick_size=12, legend_size=14
)

## Conclusiones

La implementación y evaluación del modelo GRU permitieron contrastar la capacidad de esta arquitectura no paramétrica para el pronóstico de la volatilidad del Brent. A diferencia de los modelos de la familia GARCH, la pérdida QLIKE en el conjunto de prueba fue superior (0.8383 frente al rango de 0.43 a 0.47). Asimismo, los diagnósticos de residuos fuera de muestra evidenciaron efectos ARCH en el rezago 10, lo que indicó dificultades para capturar los clústeres de volatilidad de corto plazo. 

El protocolo de búsqueda de hiperparámetros (Optuna) seleccionó una arquitectura simple para el GRU: una ventana de memoria (*look_back*) de 21 días (equivalente a un mes de negociación), 64 neuronas ocultas y una sola capa recurrente. A pesar de que el espacio de búsqueda permitía configuraciones más profundas y con mayor capacidad, el mejor desempeño en validación se obtuvo con la estructura menos compleja. Esto indicó que la red requirió poca capacidad paramétrica para procesar la dinámica del Brent, priorizando la memoria de corto plazo sobre estructuras profundas, las cuales mostraron propensión al sobreajuste.

La arquitectura GRU procesó secuencias temporales y capturó la tendencia general del mercado. Sin embargo, el modelo presentó un efecto de inercia. Debido a la naturaleza recurrente de sus compuertas y a la asimetría de la pérdida QLIKE, el optimizador adoptó un sesgo defensivo: sobreestimó la volatilidad base para evitar las penalizaciones severas por subestimación. Esto provocó que el modelo suavizara su respuesta ante los cambios de régimen, subestimando la magnitud de los shocks extremos (como el pico de 2026) y reaccionando con retraso.

El incremento en las métricas ajustadas por heterocedasticidad (HMSE y HMAE) reflejó esta inercia temporal. A pesar de este desempeño inferior en QLIKE, el modelo GRU mostró flexibilidad para adaptarse a los cambios del mercado. Al no asumir una forma matemática predefinida, la red neuronal aprendió patrones directamente de los datos. No obstante, los resultados evidenciaron que la flexibilidad no paramétrica del GRU, por sí sola, no superó la eficiencia predictiva de los modelos paramétricos. La ausencia del sesgo inductivo del GARCH (que imponía una actualización diaria de la varianza) hizo que el modelo GRU fuera más susceptible a la inercia y a las asimetrías de la función de pérdida.